In [1]:
import torch
from transformers import BertModel, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained(
    "Rostlab/prot_bert",
    do_lower_case=False
)

# Load model
model = BertModel.from_pretrained("Rostlab/prot_bert")

model = model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30, 1024, padding_idx=0)
    (position_embeddings): Embedding(40000, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-29): 30 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.0, i

In [2]:
import numpy as np
from tqdm import tqdm

def extract_and_save_embeddings(seqs, save_path, batch_size=4, max_len=512):

    all_embs = []

    with torch.inference_mode():

        for i in tqdm(range(0, len(seqs), batch_size)):

            batch = seqs[i:i+batch_size]

            # ProtBERT formatting
            batch = [
                " ".join(
                    list(
                        s.replace("U","X")
                         .replace("Z","X")
                         .replace("O","X")
                    )
                )
                for s in batch
            ]

            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_len
            )

            outputs = model(**inputs)

            # (B, L, 1024)
            hidden = outputs.last_hidden_state

            # mean pooling
            emb = hidden.mean(dim=1)  # (B, 1024)

            all_embs.append(emb.cpu().numpy())

    all_embs = np.vstack(all_embs)

    np.save(save_path, all_embs)

    return all_embs

In [3]:

from Bio import SeqIO
import pandas as pd
import numpy as np
import os
import numpy as np
import time
from itertools import product
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, matthews_corrcoef, roc_auc_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

def load_domain_file(file_path):
    sequences = []
    labels = []
    embeddings = []

    with open(file_path, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if lines[i].startswith(">"):
            seq = lines[i+1].strip()
            label = int(lines[i+2].strip())
            emb = np.array(
                list(map(float, lines[i+3].strip().split(",")))
            )

            sequences.append(seq)
            labels.append(label)
            embeddings.append(emb)

            i += 4
        else:
            i += 1

    df = pd.DataFrame({
        "sequence": sequences,
        "label": labels
    })

    X_bert = np.vstack(embeddings)
    y = np.array(labels)

    return df, X_bert, y

train_df, X_train_bert, y_train = load_domain_file("toxdl/train.fa.domain")
valid_df, X_valid_bert, y_valid = load_domain_file("toxdl/valid.fa.domain")
test_df,  X_test_bert,  y_test  = load_domain_file("toxdl/test.fa.domain")

print("Train:", X_train_bert.shape)
print("Valid:", X_valid_bert.shape)
print("Test :", X_test_bert.shape)


Train: (10084, 256)
Valid: (302, 256)
Test : (729, 256)


In [4]:
X_train_emb = extract_and_save_embeddings(
    train_df["sequence"].tolist(),
    "protbert_train.npy",
    batch_size=4   # چون CPU هستی
)

print("Train shape:", X_train_emb.shape)

X_test_emb = extract_and_save_embeddings(
    test_df["sequence"].tolist(),
    "protbert_test.npy",
    batch_size=4
)

print("Test shape:", X_test_emb.shape)


00%|████████████████████████████████████████████████████████████████████████████| 2521/2521 [5:16:33<00:00,  7.53s/it]

Train shape: (10084, 1024)


100%|████████████████████████████████████████████████████████████████████████████████| 183/183 [23:51<00:00,  7.82s/it]

Test shape: (729, 1024)


In [5]:
X_valid_emb = extract_and_save_embeddings(
    valid_df["sequence"].tolist(),
    "protbert_valid.npy",
    batch_size=4
)

print("valid shape:", X_test_emb.shape)

100%|██████████████████████████████████████████████████████████████████████████████████| 76/76 [10:15<00:00,  8.10s/it]

valid shape: (729, 1024)
